<a href="https://colab.research.google.com/github/Mitul-Marimuthu/deep-learning/blob/project1/autograd_engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

- Built a system that can differentiate any arbitrary Python expression automatically.

- A neural network trained entirely on my own gradient engine

- The exact same thing PyTorch does under the hood - just slower and scalar instead of tensor

- Know exactly what goes on when calling loss.backward() in PyTorch

In [11]:
class Value:
  def __init__(self, data, _children=(), _op='', label=''):
    self.data = data # actual number
    # how much this value affects the final loss (starts unknown)
    # filled in by backward
    self.grad = 0.0 # gradient starts at 0
    # the function that knows how to pass gradients to _prev
    self._backward = lambda: None # how to propogate gradient backward
    # builds the computation graph
    self._prev = set(_children) # what values created this one
    self._op = _op # for visualization (+, *, tanh, etc.)
    self.label = label

  def __repr__(self):
    return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"

  def __add__(self, other):
    other = other if isinstance(other, Value) else Value(other)
    out = Value(self.data + other.data, (self, other), '+')

    def _backward():
      # gradient flows equally to both inputs
      self.grad += out.grad
      other.grad += out.grad
    out._backward = _backward

    return out

  def __mul__(self, other):
    other = other if isinstance(other, Value) else Value(other)
    out = Value(self.data * other.data, (self, other), '*')

    def _backward():
      # chain rule: d(a*b)/da = b, d(a*b)/db = a
      self.grad += other.data * out.grad
      other.grad += self.data * out.grad

    out._backward = _backward

    return out

  def __neg__(self):
    return self * -1

  def __sub__(self, other):
    return self + (-other)

  def __rmul__(self, other): # handles 2 * Value, not just Value * 2
    return self * other

  def __radd__(self, other):
    return self + other

  def tanh(self):
    import math
    t = math.tanh(self.data)
    out = Value(t, (self,), 'tanh')

    # derivative of tanh is 1 - (tanh)^2
    def _backward():
      self.grad += (1 - t**2) * out.grad
    out._backward = _backward

    return out

  def relu(self):
    out = Value(max(0, self.data), (self,), 'relu')

    # if neuron was off during forward pass, no gradient flows through it
    def _backward():
      self.grad += (out.data > 0) * out.grad
    out._backward = _backward

    return out

  def backward(self):
    topo = []
    visited = set()

    # top sort neurons (inputs before outputs)
    # adds a node after all it's children (previous nodes) have been added
    def build_topo(v):
      if v not in visited:
        visited.add(v)
        for child in v._prev:
          build_topo(child)
        topo.append(v)

    build_topo(self)
    self.grad = 1.0 # starting gradient

    for v in reversed(topo):
      v._backward()


In [12]:
# sanity check for value class
a = Value(2.0, label='a')
b = Value(3.0, label='b')
c = Value(-1.0, label='c')

d = a * b + c
d.backward()

print(f"d.data = {d.data}") # expect 5.0
print(f"a.grad = {a.grad}") # expect 3.0
print(f"b.grad = {b.grad}") # expect 2.0
print(f"c.grad = {c.grad}") # expect 1.0


d.data = 5.0
a.grad = 3.0
b.grad = 2.0
c.grad = 1.0


In [14]:
import random

# This implementation is exactly how PyTorch's model.parameters() works
class Neuron:
    # weights for each INCOMING neuron
    def __init__(self, nin):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]
        self.b = Value(random.uniform(-1, 1))

    # starts the sum at self.b instead of 0
    # python trick to make the sum work over Value objects rather
    # than just plain numbers
    def __call__(self, x):
        act = sum((wi * xi for wi, xi in zip(self.w, x)), self.b)
        return act.tanh()

    # collects weights and biases
    def parameters(self):
        return self.w + [self.b]

# nin is the number of incoming weights
# nout is the number of neurons in this layer
class Layer:
    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]

    def __call__(self, x):
        return [n(x) for n in self.neurons]

    # collect weights and biases of all neurons in this layer
    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]

#MLP(3, [4, 4, 1]) — means 3 inputs, two hidden layers of
# 4 neurons each, and 1 output neuron.
# The sizes list becomes [3, 4, 4, 1] and layers are
# built between consecutive pairs.
class MLP:
    def __init__(self, nin, nouts):
        sizes = [nin] + nouts
        self.layers = [Layer(sizes[i], sizes[i+1]) for i in range(len(nouts))]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x[0] if len(x) == 1 else x

    # collects weights and biases across all neurons
    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

In [16]:
# toy binary classification dataset
xs = [
    [2.0, 3.0, -1.0],
    [3.0, -1.0, 0.5],
    [0.5, 1.0, 1.0],
    [1.0, 1.0, -1.0]
]

ys = [1.0, -1.0, -1.0, 1.0] # targets +1 or -1

model = MLP(3, [4, 4, 1])

for epoch in range(100):
    # forward pass
    ypred = [model(x) for x in xs]

    # mean squares error loss
    loss = sum((yout - ygt) * (yout - ygt) for ygt, yout in zip(ys, ypred))

    # zero gradients before backwats - critical!
    # in order to reset before the next backward pass
    # otherwise gradients fromo previous epochs added
    # onto those from current one
    # same reason why PyTorch requires optimizer.zero_grad()
    # before every backward pass
    for p in model.parameters():
        p.grad = 0.0

    # backward pass
    loss.backward()

    # gradient descent update
    for p in model.parameters():
        p.data -= 0.05 * p.grad

    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d} | Loss: {loss.data:.6f}")

print("\nFinal predictions:")
for x, y in zip(xs, ys):
    pred = model(x)
    print(f"  Input: {x} | Pred: {pred.data:+.4f} | True: {y:+.1f}")

Epoch   0 | Loss: 3.427273
Epoch  10 | Loss: 0.069310
Epoch  20 | Loss: 0.034361
Epoch  30 | Loss: 0.022624
Epoch  40 | Loss: 0.016796
Epoch  50 | Loss: 0.013327
Epoch  60 | Loss: 0.011031
Epoch  70 | Loss: 0.009402
Epoch  80 | Loss: 0.008187
Epoch  90 | Loss: 0.007247

Final predictions:
  Input: [2.0, 3.0, -1.0] | Pred: +0.9551 | True: +1.0
  Input: [3.0, -1.0, 0.5] | Pred: -0.9523 | True: -1.0
  Input: [0.5, 1.0, 1.0] | Pred: -0.9641 | True: -1.0
  Input: [1.0, 1.0, -1.0] | Pred: +0.9696 | True: +1.0
